# UNIVERSITÉ LIBRE DES PAYS DES GRANDS LACS (ULPGL)
## Faculté des Sciences et Technologies Appliquées — Master 2 Génie Électrique & Informatique
### GIN522 : Traitement d'Images & Vision par Ordinateur
---
# 🎓 PROJET INTÉGRATEUR : SYSTÈME COMPLET DE VISION
## Reconnaissance et Authentification Automatisée de Billets de Banque Congolais (CDF)

**Auteur :** Étudiant Master 2 GEI  
**Enseignant :** Ass. KAMBALE WAMUHINDO Abednego  
**Année Académique :** 2025 - 2026  
**Livrables :** Code Reproductible, Notebook Documenté, Modèle Optimal, Rapport de Synthèse  

---
### 📌 Compatibilité :
Ce notebook est **bimodal** : il s'exécute aussi bien **en local** sur CPU que sur **Google Colab** avec accélération GPU (T4/A100).


## 0. Configuration de l'Environnement et Dépendances
Si vous exécutez ce notebook sur **Google Colab**, activez le GPU (*Runtime > Change runtime type > T4 GPU*), puis exécutez la cellule ci-dessous pour installer et importer les paquets nécessaires.


In [1]:
# Installation conditionnelle pour Google Colab (decommenter si necessaire)
# !pip install torch torchvision opencv-python matplotlib seaborn scikit-learn pillow

import os
import sys
import time
import json
import random
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms

import cv2
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler

# Configuration materielle et reproductibilite
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"[OK] Environnement pret. Materiel de calcul : {device}")


[OK] Environnement pret. Materiel de calcul : cpu


## 1. Cadrage du Problème & Spécifications Métier (Semaine 1)

### 1.1 Problématique Locale et Réelle
Dans les circuits économiques en République Démocratique du Congo (RDC), la manipulation quotidienne de billets en espèces (Francs Congolais - CDF) présente des défis considérables :
* **Dépouillement et comptage manuels lents** lors des transactions sur les marchés et comptoirs de change.
* **Forte usure du papier-monnaie :** Billets froissés, souillés par l'huile/poussière, décolorés ou rafistolés au ruban adhésif.
* **Erreurs de rendu de monnaie :** Risque financier important en cas de confusion entre coupures de valeurs différentes.
* **Inclusion financière :** Absence d'outils d'assistance vocale/visuelle pour les personnes malvoyantes.

### 1.2 Formulation de la Tâche de Vision
* **Tâche choisie :** Classification supervisée multi-classes ($C = 3$).
* **Classes cibles :**
  1. `50_fc` (Coupure de 50 Francs Congolais — Dominante ocre/orangée)
  2. `500_fc` (Coupure de 500 Francs Congolais — Dominante bleue/cyan)
  3. `1000_fc` (Coupure de 1000 Francs Congolais — Dominante verte/bleue)
* **Critères de Succès Chiffrés :**
  - **Exactitude globale (*Accuracy*) $\ge 95\,\%$** sur données de test non vues.
  - **Macro-F1 Score $\ge 95\,\%$** pour garantir qu'aucune classe n'est sacrifiée.
  - **Temps d'inférence $\le 200$ ms** par image sur processeur modeste (smartphone ou Raspberry Pi).


## 2. Exploration, Audit et Hygiène des Données (Semaines 2-3)

### 2.1 Audit du Dataset
* **Total d'images :** 306 photographies prises au smartphone.
* **Résolutions :** De $810 	imes 810$ à plus de $4000 	imes 4000$ pixels (100 % 3 canaux RGB).
* **Équilibre :** 100 images de 1000 FC, 106 images de 500 FC, 100 images de 50 FC.

### 2.2 Protocole de Séparation Étanche (Sans Fuite / *No Data Leakage*)
Pour garantir une validité scientifique irréprochable, les données sont divisées avec stratification stricte et graine fixée (`seed=42`) :
- **Train (70 %) :** $214$ images
- **Validation (15 %) :** $45$ images
- **Test (15 %) :** $47$ images


In [2]:
# Verification de la repartition stratifiee
data_split_dir = r"E:\TP1_IMAGE_pROCESSING\TP8_PROJET_INTEGRATEUR_ULPGL\dataset_split_224"
if not os.path.exists(data_split_dir):
    data_split_dir = "dataset_split_224"

classes = ['1000_fc', '500_fc', '50_fc']
splits = ['train', 'val', 'test']

for s in splits:
    total_split = sum([len(os.listdir(os.path.join(data_split_dir, s, c))) for c in classes])
    details = ", ".join([f"{c}: {len(os.listdir(os.path.join(data_split_dir, s, c)))}" for c in classes])
    print(f"Jeu [{s.upper():<5}] : {total_split:3d} images ({details})")


Jeu [TRAIN] : 214 images (1000_fc: 70, 500_fc: 74, 50_fc: 70)
Jeu [VAL  ] :  45 images (1000_fc: 15, 500_fc: 15, 50_fc: 15)
Jeu [TEST ] :  47 images (1000_fc: 15, 500_fc: 17, 50_fc: 15)


## 3. Baseline Classique : Couleurs & Textures (Semaine 4)

Conformément aux exigences d'ingénierie du cours (Modules 1 à 4), nous établissons une **ligne de référence classique** sans apprentissage profond :
1. Extraction de 30 descripteurs par image :
   - Moyennes et écarts-types RGB (6)
   - Moyennes et écarts-types HSV (6)
   - Histogramme de Teinte $H$ sur 16 compartiments (16)
   - Gradient spatial de Sobel (2 descripteurs de texture)
2. Entraînement de classifieurs classiques : **KNN ($k=5$)** et **SVM (Noyau RBF)**.


In [3]:
# Performances de la baseline classique
baseline_json = r"E:\TP1_IMAGE_pROCESSING\TP8_PROJET_INTEGRATEUR_ULPGL\outputs\metrics_baseline.json"
if os.path.exists(baseline_json):
    with open(baseline_json, "r") as f:
        base_data = json.load(f)
    print(f"Meilleure Baseline : {base_data['model']}")
    print(f"  - Accuracy Test : {base_data['test_accuracy']*100:.2f}%")
    print(f"  - Macro-F1 Test : {base_data['test_macro_f1']*100:.2f}%")


Meilleure Baseline : SVM (RBF)
  - Accuracy Test : 89.36%
  - Macro-F1 Test : 89.16%


## 4. Modèle CNN par Transfer Learning (Semaines 5-6)

### 4.1 Pourquoi le Transfer Learning ?
Entraîner un réseau de neurones profond *from scratch* avec seulement 214 images conduirait inévitablement à un sur-apprentissage sévère. Nous utilisons donc l'apprentissage par transfert (*Transfer Learning*) à partir d'un réseau pré-entraîné sur **ImageNet** (1,4 million d'images).

### 4.2 Architecture et Data Augmentation
* **Backbone :** ResNet-18 (512 descripteurs, résiduel, très robuste) ou MobileNetV2.
* **Tête personnalisée :** `Dropout(p=0.3)` suivi d'une couche linéaire `Linear(512, 3)`.
* **Data Augmentation :** Rotations aléatoires ($\pm 15^\circ$), recadrages d'échelle (0.85 à 1.0), retournements horizontaux et variations colorimétriques `ColorJitter` (luminosité, contraste, saturation) pour simuler la pénombre et les éclairages de marché.


In [4]:
# Resultats optimaux de l'entrainement CNN
cnn_json = r"E:\TP1_IMAGE_pROCESSING\TP8_PROJET_INTEGRATEUR_ULPGL\outputs\metrics_cnn.json"
if os.path.exists(cnn_json):
    with open(cnn_json, "r") as f:
        cnn_data = json.load(f)
    print(f"Modele : {cnn_data['architecture'].upper()}")
    print(f"  - Epoque Optimale          : {cnn_data['best_epoch']}/15")
    print(f"  - Validation Loss Minimale : {cnn_data['best_val_loss']:.4f}")
    print(f"  - Validation Accuracy      : {cnn_data['best_val_acc']*100:.2f}%")


Modele : RESNET18
  - Epoque Optimale          : 14/15
  - Validation Loss Minimale : 0.0279
  - Validation Accuracy      : 100.00%


## 5. Évaluation Honnête sur le Test & Autopsie Critique (Semaine 7)

### 5.1 Comparatif Scientifique : Baseline vs CNN


In [5]:
# Comparatif final sur le jeu de test
metrics_final_path = r"E:\TP1_IMAGE_pROCESSING\TP8_PROJET_INTEGRATEUR_ULPGL\outputs\metrics_comparatif_final.json"
if os.path.exists(metrics_final_path):
    with open(metrics_final_path, "r") as f:
        final_data = json.load(f)
    
    print("=" * 60)
    print("BILAN COMPARATIF SUR LES 47 IMAGES DE TEST (INCONNUES)")
    print("=" * 60)
    print(f"Baseline ({final_data['baseline']['name']}) : Accuracy = {final_data['baseline']['accuracy']*100:.2f}% | F1 = {final_data['baseline']['macro_f1']*100:.2f}%")
    print(f"CNN (ResNet18)         : Accuracy = {final_data['cnn_resnet18']['accuracy']*100:.2f}% | F1 = {final_data['cnn_resnet18']['macro_f1']*100:.2f}%")
    print("-" * 60)
    print(f"--> Gain apporte par le CNN : +{final_data['gain_accuracy_points']:.2f} points de pourcentage !")


BILAN COMPARATIF SUR LES 47 IMAGES DE TEST (INCONNUES)
Baseline (SVM (RBF)) : Accuracy = 89.36% | F1 = 89.16%
CNN (ResNet18)         : Accuracy = 100.00% | F1 = 100.00%
------------------------------------------------------------
--> Gain apporte par le CNN : +10.64 points de pourcentage !


## 6. Déploiement & Démonstration Interactive (Semaine 8)

Fonction d'inférence autonome prête pour l'intégration mobile ou embarquée :


In [6]:
def predire_billet(image_path, model_path=r"E:\TP1_IMAGE_pROCESSING\TP8_PROJET_INTEGRATEUR_ULPGL\modelsest_resnet18_billets.pth"):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    class_names = ['1000_fc', '500_fc', '50_fc']
    
    model = models.resnet18(weights=None)
    model.fc = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(model.fc.in_features, 3))
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    t0 = time.time()
    img = Image.open(image_path).convert("RGB")
    tensor = tf(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        out = model(tensor)
        probs = nn.Softmax(dim=1)(out).cpu().numpy()[0]
        
    latency = (time.time() - t0) * 1000
    idx = np.argmax(probs)
    conf = probs[idx]
    
    print(f"Fichier        : {os.path.basename(image_path)}")
    print(f"Classe Predite : {class_names[idx].upper()} ({conf*100:.2f}%)")
    print(f"Temps Inférence: {latency:.1f} ms")
    return class_names[idx], conf

# Test live
demo_img = r"E:\TP1_IMAGE_pROCESSING\TP8_PROJET_INTEGRATEUR_ULPGL\dataset_split_224	est@0_fc@0_fc_004.jpg"
if os.path.exists(demo_img):
    predire_billet(demo_img)


Fichier        : 1000_fc_004.jpg
Classe Predite : 1000_FC (99.99%)
Temps Inférence: 122.2 ms


## 7. Limites, Usages Responsables et Éthique de l'Ingénieur

Conformément à la grille de notation officielle (diapositive 13 & 24) :
1. **Dérive de Domaine (*Domain Shift*) :**
   Le modèle a été entraîné sur un jeu de données de 306 images avec des arrière-plans maîtrisés. Dans des conditions de marché réelles (billets pliés en boule, salissures d'huile de palme, déchirures recollées), les performances peuvent chuter. 
   *Remède d'ingénieur :* Mise en place d'un seuil de rejet (*Reject Option* / confiance $< 60\,\%$) pour inviter l'utilisateur à replacer le billet correctement.

2. **Absence de Détection de Faux Billets :**
   Ce modèle est un classifieur de coupures, **non un détecteur de fausse monnaie**. Il ne doit pas être vendu comme un outil de sécurité bancaire sans intégrer des capteurs ultraviolets (UV) ou infrarouges (IR).

3. **Protection de la Vie Privée :**
   L'application traite uniquement la zone du billet et ne stocke aucune photo de visage ni métadonnée personnelle des clients.

4. **Déploiement Embarqué Hors-Ligne :**
   Avec ResNet18 (46 Mo) ou MobileNetV2 (14 Mo), le système fonctionne intégralement **en local sans connexion Internet**, ce qui est indispensable pour les zones à connectivité instable.
